####Funciones simples de agregación

In [0]:
%run "../includes/configurations"

In [0]:
movie_df = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
from pyspark.sql.functions import count, countDistinct, sum, max, min, avg

In [0]:
movie_df.select(count("*")).show()

In [0]:
movie_df.select(count("year_release_date")).show()

In [0]:
movie_df.select(countDistinct("year_release_date")).show()

In [0]:
movie_df.select(sum("budget")).display()

In [0]:
movie_df.filter("year_release_date=2016") \
    .select(sum("budget"), count("movie_id")) \
    .withColumnRenamed("sum(budget)", "total_budget") \
    .withColumnRenamed("count(movie_id)", "total_movies") \
    .display()

In [0]:
display(movie_df)

###Group By

In [0]:
movie_df \
        .groupBy("year_release_date") \
        .agg(
            sum("budget").alias("total_budget"),
            avg("budget").alias("avg_budget"),
            max("budget"),
            min("budget"),
            count("movie_id")
        ) \
        .display()

###Window Functions

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc, dense_rank

In [0]:
movie_df.select("title", "budget", "year_release_date") \
        .filter("year_release_date is not null") \
        .withColumn("rank", rank().over(Window.partitionBy("year_release_date").orderBy(desc("budget")))) \
        .withColumn("desc_rank", dense_rank().over(Window.partitionBy("year_release_date").orderBy(desc("budget")))) \
        .display()